# Exploration des achats de produits phytosanitaires par département

Données : quantités totales de produits phytosanitaires achetées, par département et par substance.
Placer le fichier `quantite-totale-de-produit-phytosanitaire-achetee-en-france-departement.csv`
dans `data/raw/` avant d'exécuter le notebook.

In [ ]:
import pandas as pd

In [ ]:
import os
from pathlib import Path

# Se placer à la racine du dépôt (le notebook est dans notebooks/exploration/)
for _parent in [Path.cwd(), *Path.cwd().parents]:
    if (_parent / "src").is_dir():
        os.chdir(_parent)
        break

chemin_csv = Path("data/raw/quantite-totale-de-produit-phytosanitaire-achetee-en-france-departement.csv")
print("Fichier trouvé :", chemin_csv.exists())


In [ ]:
df = pd.read_csv(chemin_csv, dtype={"geocode_departement": str})  # garde "01" et "2A" en texte

print("Dimensions du jeu de données :", df.shape)
print("\nNoms des colonnes :")
print(df.columns.tolist())

display(df.head())

In [ ]:
# 1. Types des colonnes et mémoire utilisée
df.info()

# 2. Nombre de valeurs manquantes par colonne
print("\nValeurs manquantes :")
display(df.isna().sum().to_frame(name="valeurs_manquantes"))

# 3. Statistiques descriptives des colonnes numériques
print("\nStatistiques de la colonne valeur :")
display(df["valeur"].describe())

In [ ]:
# Convertir date_mesure en véritable date Pandas
df["date_mesure"] = pd.to_datetime(df["date_mesure"])

# Examiner les années disponibles
print("Période couverte :")
print("Première date :", df["date_mesure"].min())
print("Dernière date :", df["date_mesure"].max())
print("\nNombre de lignes par année :")
display(df["date_mesure"].dt.year.value_counts().sort_index())

# Vérifier les valeurs négatives et les zéros
print("\nValeurs négatives :", (df["valeur"] < 0).sum())
print("Valeurs nulles :", (df["valeur"] == 0).sum())

# Examiner les 10 valeurs les plus élevées
print("\nLes 10 valeurs les plus élevées :")
display(
    df.nlargest(10, "valeur")[
        ["date_mesure", "geocode_departement",
         "libelle_departement", "substance", "valeur"]
    ]
)

In [ ]:
# 1. Nombre de départements et liste des codes
print("Nombre de codes départementaux :",
      df["geocode_departement"].nunique())

print("\nCodes départementaux :")
print(sorted(df["geocode_departement"].unique()))

# 2. Vérifier les substances
print("\nNombre de substances différentes :",
      df["substance"].nunique())

# 3. Vérifier les catégories de danger
print("\nRépartition des catégories de danger :")
display(df["danger_substance"].value_counts())

# 4. Vérifier les modalités du biocontrôle
print("\nRépartition du biocontrôle :")
display(df["biocontrole"].value_counts())

In [ ]:
# 1. Extraire l'année
df["annee"] = df["date_mesure"].dt.year

# 2. Vérifier les départements présents par année
print("Nombre de départements par année :")
display(
    df.groupby("annee")["geocode_departement"]
      .nunique()
      .rename("nombre_departements")
)

# 3. Rechercher les doublons exacts sur les variables disponibles
colonnes_observation = [
    "date_mesure",
    "geocode_departement",
    "biocontrole",
    "danger_substance",
    "substance"
]

print(
    "Doublons sur les variables descriptives :",
    df.duplicated(subset=colonnes_observation).sum()
)

# 4. Afficher quelques lignes concernant les départements d'outre-mer
codes_outre_mer = ["971", "972", "973", "974", "976"]

display(
    df[df["geocode_departement"].isin(codes_outre_mer)]
      .head(10)
)

In [ ]:
# 1. Examiner les modalités des variables catégorielles
print("Catégories de danger :")
print(sorted(df["danger_substance"].unique()))

print("\nModalités du biocontrôle :")
print(sorted(df["biocontrole"].unique()))

# 2. Examiner les départements métropolitains et ultramarins
codes = sorted(df["geocode_departement"].unique())

codes_outre_mer = {"971", "972", "973", "974", "976"}
codes_metropole = [c for c in codes if c not in codes_outre_mer]

print("\nNombre de codes hors outre-mer :", len(codes_metropole))
print("Codes hors outre-mer :", codes_metropole)

# 3. Vérifier si une substance apparaît plusieurs fois
# dans un même département et une même année
nb_lignes_par_groupe = df.groupby(
    ["geocode_departement", "annee", "substance"]
).size()

print(
    "\nGroupes département-année-substance avec plusieurs lignes :",
    (nb_lignes_par_groupe > 1).sum()
)

display(nb_lignes_par_groupe[nb_lignes_par_groupe > 1].head(10))

In [ ]:
# Pour chaque département, année et substance,
# compter les lignes et les modalités des deux variables
analyse_doublons = (
    df.groupby(["geocode_departement", "annee", "substance"])
      .agg(
          nb_lignes=("valeur", "size"),
          nb_biocontrole=("biocontrole", "nunique"),
          nb_categories_danger=("danger_substance", "nunique")
      )
)

# Garder uniquement les groupes ayant plusieurs lignes
groupes_multiples = analyse_doublons[
    analyse_doublons["nb_lignes"] > 1
]

print("Groupes ayant plusieurs lignes :", len(groupes_multiples))
print(
    "Groupes où biocontrole varie :",
    (groupes_multiples["nb_biocontrole"] > 1).sum()
)
print(
    "Groupes où la catégorie de danger varie :",
    (groupes_multiples["nb_categories_danger"] > 1).sum()
)

display(groupes_multiples.head(15))

In [ ]:
display(
    df[
        (df["geocode_departement"] == "01")
        & (df["annee"] == 2013)
        & (df["substance"] == "pyrethrines")
    ][
        [
            "date_mesure",
            "geocode_departement",
            "annee",
            "substance",
            "biocontrole",
            "danger_substance",
            "valeur"
        ]
    ]
)

In [ ]:
# Conserver toutes les années pour l'exploration
df_agrege = (
    df.groupby(
        [
            "geocode_departement",
            "libelle_departement",
            "annee",
            "substance",
            "danger_substance"
        ],
        as_index=False
    )
    .agg(
        quantite_totale=("valeur", "sum")
    )
)

print("Période :", df_agrege["annee"].min(),
      "à", df_agrege["annee"].max())
print("Dimensions avant agrégation :", df.shape)
print("Dimensions après agrégation :", df_agrege.shape)

display(df_agrege.head())

In [ ]:
# Vérifier l'unicité des combinaisons après agrégation
cles = [
    "geocode_departement",
    "annee",
    "substance",
    "danger_substance"
]

print("Nombre de doublons restants :", df_agrege.duplicated(subset=cles).sum())

print("\nNombre de lignes :", len(df_agrege))
print("Nombre de départements :", df_agrege["geocode_departement"].nunique())
print("Nombre d'années :", df_agrege["annee"].nunique())
print("Nombre de substances :", df_agrege["substance"].nunique())

print("\nValeurs manquantes :")
display(df_agrege.isna().sum().to_frame(name="valeurs_manquantes"))

In [ ]:
# Somme des quantités par département sur toute la période
quantites_departements = (
    df_agrege.groupby(
        ["geocode_departement", "libelle_departement"],
        as_index=False
    )
    .agg(quantite_totale=("quantite_totale", "sum"))
    .sort_values("quantite_totale", ascending=False)
)

# Afficher les 10 premiers départements
display(quantites_departements.head(10))

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

url_geojson = (
    "https://raw.githubusercontent.com/"
    "gregoiredavid/france-geojson/master/departements.geojson"
)

departements = gpd.read_file(url_geojson)

print("Nombre de départements :", len(departements))
print("Colonnes :", departements.columns.tolist())

departements.head()

In [ ]:
# Préparer les codes pour la jointure
quantites_departements["code"] = (
    quantites_departements["geocode_departement"].astype(str).str.zfill(2)
)

departements["code"] = departements["code"].astype(str).str.zfill(2)

# Associer les quantités aux contours géographiques
carte = departements.merge(
    quantites_departements[["code", "quantite_totale"]],
    on="code",
    how="left"
)

print("Nombre de départements :", len(carte))
print("Départements sans quantité :", carte["quantite_totale"].isna().sum())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10))

carte.plot(
    column="quantite_totale",
    cmap="YlOrRd",
    linewidth=0.4,
    edgecolor="white",
    legend=True,
    ax=ax,
    legend_kwds={
        "label": "Quantité cumulée sur 2013–2023",
        "shrink": 0.6
    }
)

ax.set_title(
    "Quantités de produits phytosanitaires achetées par département\n"
    "France métropolitaine — 2013 à 2023",
    fontsize=15,
    pad=15
)

ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

fig, ax = plt.subplots(figsize=(12, 10))

carte.plot(
    column="quantite_totale",
    cmap="YlOrRd",
    norm=LogNorm(
        vmin=carte.loc[carte["quantite_totale"] > 0, "quantite_totale"].min(),
        vmax=carte["quantite_totale"].max()
    ),
    linewidth=0.4,
    edgecolor="white",
    legend=True,
    ax=ax,
    legend_kwds={
        "label": "Quantité cumulée (échelle logarithmique)",
        "shrink": 0.6
    }
)

ax.set_title(
    "Quantités de produits phytosanitaires achetées par département\n"
    "France métropolitaine — 2013 à 2023 (échelle logarithmique)",
    fontsize=15,
    pad=15
)

ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
quantites_annuelles = (
    df_agrege
    .groupby("annee", as_index=False)
    .agg(quantite_totale=("quantite_totale", "sum"))
    .sort_values("annee")
)

display(quantites_annuelles)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    quantites_annuelles["annee"],
    quantites_annuelles["quantite_totale"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Évolution annuelle des achats de produits phytosanitaires (2013–2023)"
)
ax.set_xlabel("Année")
ax.set_ylabel("Quantité totale (en millions d'unités de valeur)")

# Afficher l'axe vertical en millions
ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x / 1e6:.1f}")
)

ax.set_xticks(quantites_annuelles["annee"])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import random

random.seed(0)  # même tirage à chaque exécution

substances_selectionnees = random.sample(
    df_agrege["substance"].dropna().unique().tolist(),
    10
)

df_substances = df_agrege[
    df_agrege["substance"].isin(substances_selectionnees)
].copy()

print(substances_selectionnees)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_evolution = (
    df_substances
    .groupby(["annee", "substance"], as_index=False)["quantite_totale"]
    .sum()
)

df_evolution.head()

In [ ]:
df_pivot = (
    df_evolution
    .pivot(index="annee", columns="substance", values="quantite_totale")
    .fillna(0)
)

df_pivot.plot(
    kind="area",
    stacked=True,
    figsize=(14, 7),
    alpha=0.8
)

plt.title("Évolution des achats de 10 substances phytosanitaires (2013–2023)")
plt.xlabel("Année")
plt.ylabel("Quantité totale achetée")
plt.legend(title="Substance", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
quantites_substances = (
    df_substances
    .groupby("substance")["quantite_totale"]
    .sum()
    .sort_values()
)

plt.figure(figsize=(10, 6))

quantites_substances.plot(
    kind="barh",
    color="steelblue"
)

plt.title("Quantité totale achetée par substance (2013–2023)")
plt.xlabel("Quantité totale achetée")
plt.ylabel("Substance")
plt.tight_layout()
plt.show()

In [ ]:
quantites_substances = (
    df_substances
    .groupby("substance")["quantite_totale"]
    .sum()
    .sort_values()
)

plt.figure(figsize=(10, 6))

quantites_substances.plot(
    kind="barh",
    color="steelblue",
    logx=True
)

plt.title("Quantité totale achetée par substance (2013–2023)")
plt.xlabel("Quantité totale achetée (échelle logarithmique)")
plt.ylabel("Substance")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

df_heatmap = (
    df_evolution
    .pivot(index="substance", columns="annee", values="quantite_totale")
    .fillna(0)
)

plt.figure(figsize=(14, 7))

sns.heatmap(
    df_heatmap,
    cmap="YlOrRd",
    linewidths=0.3,
    cbar_kws={"label": "Quantité totale achetée"}
)

plt.title("Achats de produits phytosanitaires par substance et par année")
plt.xlabel("Année")
plt.ylabel("Substance")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import seaborn as sns

df_heatmap = (
    df_evolution
    .pivot(index="substance", columns="annee", values="quantite_totale")
    .fillna(0)
)

plt.figure(figsize=(14, 7))

sns.heatmap(
    np.log10(df_heatmap + 1),
    cmap="YlOrRd",
    linewidths=0.3,
    cbar_kws={"label": "log₁₀(quantité + 1)"}
)

plt.title("Achats de produits phytosanitaires par substance et par année")
plt.xlabel("Année")
plt.ylabel("Substance")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))

sns.boxplot(
    data=df_substances,
    x="substance",
    y="quantite_totale",
    hue="substance",
    legend=False
)

plt.yscale("log")
plt.title("Distribution des achats par substance et par département")
plt.xlabel("Substance")
plt.ylabel("Quantité achetée par département (échelle logarithmique)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    nrows=5,
    ncols=2,
    figsize=(15, 18),
    sharex=True
)

for ax, substance in zip(axes.flat, substances_selectionnees):
    donnees = (
        df_evolution[df_evolution["substance"] == substance]
        .sort_values("annee")
    )

    ax.plot(
        donnees["annee"],
        donnees["quantite_totale"],
        marker="o"
    )

    ax.set_title(substance)
    ax.set_xlabel("Année")
    ax.set_ylabel("Quantité achetée")
    ax.grid(alpha=0.3)

plt.suptitle(
    "Évolution annuelle des achats par substance (2013–2023)",
    fontsize=16,
    y=1.01
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))

sns.violinplot(
    data=df_substances,
    x="substance",
    y="quantite_totale",
    hue="substance",
    legend=False,
    cut=0
)

plt.yscale("symlog")
plt.title("Distribution des achats par substance")
plt.xlabel("Substance")
plt.ylabel("Quantité achetée (échelle symlog)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))

sns.ecdfplot(
    data=df_substances,
    x="quantite_totale",
    hue="substance"
)

plt.title("Fonction de répartition empirique des achats par substance")
plt.xlabel("Quantité achetée")
plt.ylabel("Proportion cumulée des observations")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_concentration = (
    df_substances
    .groupby(
        ["geocode_departement", "libelle_departement", "substance"],
        as_index=False
    )["quantite_totale"]
    .sum()
)
def calcul_gini(valeurs):
    valeurs = np.asarray(valeurs, dtype=float)

    if np.any(valeurs < 0):
        raise ValueError("Le calcul du Gini nécessite des valeurs positives ou nulles.")

    if valeurs.sum() == 0:
        return np.nan

    valeurs = np.sort(valeurs)
    n = len(valeurs)

    rangs = np.arange(1, n + 1)

    return (2 * np.sum(rangs * valeurs)) / (n * valeurs.sum()) - (n + 1) / n


indices_gini = (
    df_concentration
    .groupby("substance")["quantite_totale"]
    .apply(calcul_gini)
    .sort_values(ascending=False)
)

indices_gini
plt.figure(figsize=(10, 6))

indices_gini.sort_values().plot(
    kind="barh",
    color="steelblue"
)

plt.title("Concentration géographique des achats par substance")
plt.xlabel("Indice de Gini")
plt.ylabel("Substance")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(10, 8))

for substance in substances_selectionnees:
    quantites = (
        df_concentration[
            df_concentration["substance"] == substance
        ]["quantite_totale"]
        .sort_values()
        .to_numpy()
    )

    if len(quantites) == 0 or quantites.sum() == 0:
        continue

    part_departements = np.arange(len(quantites) + 1) / len(quantites)

    part_quantites = np.insert(
        np.cumsum(quantites) / quantites.sum(),
        0,
        0
    )

    plt.plot(
        part_departements,
        part_quantites,
        label=substance
    )

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="black",
    label="Égalité parfaite"
)

plt.title("Courbes de Lorenz des achats par substance")
plt.xlabel("Part cumulée des départements")
plt.ylabel("Part cumulée des quantités achetées")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.axis("equal")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_correlation = (
    df_concentration
    .pivot(
        index="geocode_departement",
        columns="substance",
        values="quantite_totale"
    )
    .fillna(0)
)

matrice_correlation = df_correlation.corr(method="pearson")

plt.figure(figsize=(11, 9))

sns.heatmap(
    matrice_correlation,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True
)

plt.title("Corrélation des quantités achetées entre substances")
plt.tight_layout()
plt.show()

In [ ]:
df_classement = df_evolution[
    df_evolution["annee"].isin([2013, 2023])
].copy()

df_classement["rang"] = (
    df_classement
    .groupby("annee")["quantite_totale"]
    .rank(method="min", ascending=False)
)

df_classement.head()


df_classement = df_evolution.copy()

df_classement["rang"] = (
    df_classement
    .groupby("annee")["quantite_totale"]
    .rank(method="min", ascending=False)
)

df_rangs = (
    df_classement
    .pivot(index="annee", columns="substance", values="rang")
    [substances_selectionnees]
)

plt.figure(figsize=(13, 7))

for substance in substances_selectionnees:
    plt.plot(
        df_rangs.index,
        df_rangs[substance],
        marker="o",
        linewidth=2,
        label=substance
    )

plt.gca().invert_yaxis()
plt.xticks(range(2013, 2024))
plt.yticks(range(1, len(substances_selectionnees) + 1))
plt.xlabel("Année")
plt.ylabel("Rang (1 = quantité la plus élevée)")
plt.title("Évolution annuelle du classement des 10 substances")
plt.grid(alpha=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
df_danger = (
    df_substances
    .groupby(["annee", "danger_substance"], as_index=False)["quantite_totale"]
    .sum()
)

df_danger_pivot = (
    df_danger
    .pivot(index="annee", columns="danger_substance", values="quantite_totale")
    .fillna(0)
)

df_danger_pivot.plot(
    kind="line",
    marker="o",
    figsize=(13, 7),
    linewidth=2
)

plt.title("Évolution des achats selon la classification de danger (2013–2023)")
plt.xlabel("Année")
plt.ylabel("Quantité totale achetée")
plt.xticks(range(2013, 2024))
plt.legend(title="Classification de danger", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_tendance = (
    df_evolution
    .groupby("annee", as_index=False)["quantite_totale"]
    .sum()
)

df_tendance


import numpy as np
import matplotlib.pyplot as plt

x = df_tendance["annee"].to_numpy()
y = df_tendance["quantite_totale"].to_numpy()

# Ajustement d'une droite sur l'ensemble de la période
coefficients = np.polyfit(x, y, 1)
tendance = np.polyval(coefficients, x)

plt.figure(figsize=(12, 6))

plt.plot(x, y, marker="o", linewidth=2, label="Quantités observées")
plt.plot(x, tendance, linestyle="--", linewidth=2, label="Tendance linéaire")

plt.xticks(range(2013, 2024))
plt.xlabel("Année")
plt.ylabel("Quantité totale achetée")
plt.title("Évolution des achats de 10 substances phytosanitaires (2013–2023)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_tendance = df_evolution.copy()

df_tendance = df_tendance.sort_values(["substance", "annee"])

df_tendance["variation_absolue"] = (
    df_tendance.groupby("substance")["quantite_totale"].diff()
)

df_tendance["variation_pct"] = (
    df_tendance.groupby("substance")["quantite_totale"].pct_change() * 100
)

df_tendance

In [ ]:
df_tendance = df_evolution.copy()

df_tendance = df_tendance.sort_values(["substance", "annee"])

df_tendance["variation_absolue"] = (
    df_tendance.groupby("substance")["quantite_totale"].diff()
)

df_tendance["variation_pct"] = (
    df_tendance.groupby("substance")["quantite_totale"].pct_change() * 100
)

df_tendance

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    np.log10(df_substances["quantite_totale"] + 1),
    bins=40,
    edgecolor="black",
    alpha=0.75
)

plt.title("Distribution logarithmique des quantités achetées")
plt.xlabel("log₁₀(quantité + 1)")
plt.ylabel("Nombre d'observations")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13, 7))

for substance in substances_selectionnees:
    valeurs = df_substances.loc[
        df_substances["substance"] == substance,
        "quantite_totale"
    ]

    valeurs_log = np.log10(valeurs + 1)

    sns.kdeplot(
        x=valeurs_log,
        label=substance,
        linewidth=2
    )

plt.title("Distribution des quantités achetées par substance (KDE)")
plt.xlabel("log₁₀(quantité + 1)")
plt.ylabel("Densité")
plt.legend(title="Substance", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()